### Load needed libraries

In [ ]:
import os
import shutil
import scanpy as sc
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib
import copy
from matplotlib import pyplot as plt
from matplotlib import cm as cm
%matplotlib inline
from scipy import stats as sp_stats
import warnings
from datetime import datetime

sc.settings.n_jobs = 32
sc.set_figure_params(scanpy=True, dpi=100, dpi_save=500, frameon=False, vector_friendly=True, figsize=(10,10), format='png')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams["axes.grid"] = False


In [ ]:
from validation.validation_functions import get_shared_dir, get_result_dir, TOPICS_DATA_H5_PATH, CELL_TYPE_TO_GROUP

# Downloaded data from SEA-AD project
SEAD_DIR = os.path.join(get_shared_dir, "SeaAD")
CELL_TYPE_TO_GROUP = {
    "inhibitory": "neuronal/inhibitory",
    "cux2p": "neuronal/excitatory/cux2plus",
    "cux2m": "neuronal/excitatory/cux2minus",
    "microglia": "glia/microglia",
    "astrocytes": "glia/astrocytes",
    "oligo": "glia/oligodendrocytes",
    "opcs": "glia/opcs",
}


### Load the needed data files

In [ ]:
adata = sc.read_h5ad("./validation/sead/SEAAD_MTG_RNAseq_final-nuclei-without-reference.2024-02-13.h5ad")


In [ ]:
adata

In [ ]:
from collections import Counter
Counter(adata.obs['Cognitive Status'])
# Donor ID, Sex, PMI, Overall AD neuropathological Change, Cognitive Status, Last MMSE Score, Class, Subclass, Supertype, Severely Affected Donor

In [ ]:
Counter(adata.obs['Subclass'])


In [ ]:
cell_type_configuration = {
    "inhibitory": {"Class": "Neuronal: GABAergic"},
    "excitatory": {"Class": "Neuronal: Glutamatergic"},
    "microglia": {"Subclass": "Microglia-PVM"},
    "astrocytes": {"Subclass": "Astrocyte"},
    "opcs": {"Subclass": "OPC"},
    "oligo": {"Subclass": "Oligodendrocyte"},
}

### Cell type and run options

**To run this notebook for a specific cell type:**

- **Option A (Papermill, from project root):**  
  `papermill validation/sead/SEA_AD_gene_signature.ipynb output.ipynb -p cell_type <cell_type>`  
  Example: `papermill validation/sead/SEA_AD_gene_signature.ipynb output_inhibitory.ipynb -p cell_type inhibitory`

- **Option B (Manual):** In the **next code cell**, set `cell_type` to one of:
  - `"inhibitory"`, `"excitatory"`, `"microglia"`, `"astrocytes"`, `"oligo"`, `"opcs"`

`project` and `root_plots_dir` stay `'seaad'` and `"./Plots/seaad"` unless you override them via papermill (e.g. `-p root_plots_dir ./OtherPlots`).

In [ ]:
import pathlib

cell_type = "inhibitory"

project = 'seaad'
root_plots_dir = os.path.join(get_result_dir(), "seaad")

plots_dir = f"{root_plots_dir}/{cell_type}"
adata_path = os.path.join(plots_dir, f"{cell_type}_adata.h5ad")
try:
    os.mkdir(root_plots_dir)
except FileExistsError:
    pass
try:
    os.mkdir(plots_dir)
except FileExistsError:
    pass

print(f"cell_type: {cell_type} adata path: {adata_path}")

In [ ]:
if cell_type_configuration[cell_type].get('Class'):
    sub_adata = adata[adata.obs['Class'] == cell_type_configuration[cell_type]['Class']].copy()
else:
    sub_adata = adata[adata.obs['Subclass'] == cell_type_configuration[cell_type]['Subclass']].copy()
print(f"Writing adata to {adata_path}")
sub_adata.write_h5ad(adata_path)

In [ ]:
sub_adata = sc.read_h5ad(adata_path)


In [ ]:

import h5py
import pandas as pd

if cell_type == "excitatory":
    with h5py.File(TOPICS_DATA_H5_PATH, "r") as h5file:
        des_cux2p = pd.DataFrame(h5file[CELL_TYPE_TO_GROUP["cux2p"] + '/de'][()])
    with h5py.File(TOPICS_DATA_H5_PATH, "r") as h5file:
        des_cux2m = pd.DataFrame(h5file[CELL_TYPE_TO_GROUP["cux2m"] + '/de'][()])
    des = pd.concat([des_cux2p, des_cux2m], ignore_index=True)
    if "cell.type" in des.columns:
        des["topic"] = des["cell.type"].astype(str) + "_" + des["topic"].astype(str)
else:
    print(CELL_TYPE_TO_GROUP[cell_type])
    with h5py.File(TOPICS_DATA_H5_PATH, "r") as h5file:
        des = pd.DataFrame(h5file[CELL_TYPE_TO_GROUP[cell_type] + '/de'][()])

# convert gene and topic to string
des["gene"] = des["gene"].astype(str)
des["topic"] = des["topic"].astype(str)
genes_by_topic = (
    des.groupby('topic')['gene']
    .apply(list)
    .to_dict()
)

In [ ]:
neuronal_topics_to_annotation = {
  "decreasing_early":("inh_k1","cux2p_k9", "cux2m_k4"),
  "decreasing_late":("inh_k7","cux2p_k10", "cux2m_k8"),
  "increasing_early":("inh_k10","cux2p_k11", "cux2m_k3"),
  "increasing_late":("inh_k8","cux2p_k4", "cux2m_k9")
}


In [ ]:
if cell_type == 'excitatory':
    for union_name, topic_names in neuronal_topics_to_annotation.items():
    # Apply convert_topic_name to each key in topic_names_raw
        print(topic_names)
        # Only keep topic nabmes that exist in genes_by_topic
        topic_names = [name for name in topic_names if name in genes_by_topic]
        if len(topic_names) > 0:
            # Gather all genes from selected topics, flatten, take unique
            union_genes = list(set(gene for t in topic_names for gene in genes_by_topic[t]))
            genes_by_topic[union_name] = union_genes


    genes_by_topic = {k: genes_by_topic[k] for k in genes_by_topic if k in ['decreasing_early', 'decreasing_late', 'increasing_early', 'increasing_late']}

In [ ]:
# Create a list of genes per topic that don't appear in other topics
exclusive_genes_by_topic = {}
for topic, genes in genes_by_topic.items():
    other_genes = set(
        gene
        for other_topic, other_genes_list in genes_by_topic.items() 
        if other_topic != topic
        for gene in other_genes_list
    )
    exclusive = [gene for gene in genes if gene not in other_genes]
    exclusive_genes_by_topic[topic] = exclusive



In [ ]:
import validation
import h5py
from validation.validation_functions import compute_coherence_multiple

SHOULD_COMPUTE_COHERENCE = True
if cell_type in ("inhibitory", "excitatory") and SHOULD_COMPUTE_COHERENCE:
    n_cores = int(os.environ['SLURM_CPUS_ON_NODE'])

    if cell_type == "inhibitory":
        programs_to_check = [f"k{i}" for i in (1,7,8,10)]
        gene_sets_to_test = {k:v for k,v in exclusive_genes_by_topic.items() if k in programs_to_check}
    else:
        gene_sets_to_test = genes_by_topic.copy()
    # Get expression matrix once (memory efficient)
    expr_matrix = sub_adata.X.T

    # Compute coherence for all programs
    coherence_results = compute_coherence_multiple(expr_matrix, gene_sets_to_test, gene_names=sub_adata.var_names, n_permutations=100, n_cores=n_cores, temp_folder=os.path.join(get_result_dir(), "tmp", "joblib"))
    coherence_results.to_pickle(os.path.join(plots_dir, "coherence_results.pickle"))
                    # Convert strings to fixed-length bytes

    coherence_results['gene_set'] = coherence_results['gene_set'].astype(str)
    coherence_results['gene_set'] = np.array([str(x).encode('utf-8') for x in coherence_results['gene_set']], dtype='S')


    with h5py.File(TOPICS_DATA_H5_PATH, "a") as h5f:
        group_path = f"{CELL_TYPE_TO_GROUP[cell_type]}/{project}"
        h5f.create_dataset(f"{group_path}/coherence_results", data=coherence_results.drop(columns=['bg_means_cor'], inplace=False).to_records(index=False))
        bg_means_list = coherence_results['bg_means_cor'].tolist()
        # Stack into 2D array
        bg_means_2d = np.vstack([np.array(x) for x in bg_means_list])
        h5f.create_dataset(f"{group_path}/coherence_results_bg_means", data=bg_means_2d)



In [ ]:
for topic, genes in genes_by_topic.items():
    print(f"Scoring topic {topic} with {len(genes)} genes")
    genes_symbol = sub_adata.var_names[sub_adata.var_names.isin(genes)].tolist()
    print(f"Found {len(genes_symbol)} genes in adata.var_names")
    sc.tl.score_genes(sub_adata, genes_symbol, score_name=f"Signature_{topic}", use_raw=False)

In [ ]:
for topic, genes in exclusive_genes_by_topic.items():
    print(f"Scoring topic {topic} with {len(genes)} exclusive genes")
    genes_symbol = sub_adata.var_names[sub_adata.var_names.isin(genes)].tolist()
    print(f"Found {len(genes_symbol)} genes in adata.var_names")
    sc.tl.score_genes(sub_adata, genes_symbol, score_name=f"SignatureUnique_{topic}", use_raw=False)
    print(f"Scored topic {topic} with {len(genes_symbol)} exclusive genes")

In [ ]:
import os

# Extract all columns from sub_adata.obs that start with "Signature"
signature_cols = [col for col in sub_adata.obs.columns if col.startswith("Signature")]
signature_df = sub_adata.obs[signature_cols]

# Save as pickle, equivalent to RDS in R
signature_pickle_path = os.path.join(plots_dir, "Signature.pickle")
signature_df.to_pickle(signature_pickle_path)
signature_df.to_csv(os.path.join(plots_dir, "Signature.csv"))

In [ ]:
import json

pathway_file_format = os.path.join(get_result_dir(), 'topic_space', 'Pathways_genes_{cell_type}.json')

if cell_type == 'excitatory':
    with open(pathway_file_format.format(cell_type='cux2p')) as f:
        pathways = json.load(f)

    with open(pathway_file_format.format(cell_type='cux2m')) as f:
        new_pathways = json.load(f)
        for k, v in new_pathways.items():
            if k in pathways:
                # Union of gene lists
                pathways[k] = list(set(pathways[k]).union(set(v)))
            else:
                pathways[k] = v


else:
    with open(pathway_file_format.format(cell_type=cell_type)) as f:
        pathways = json.load(f)


if cell_type in ["inhibitory", "excitatory"]:
    excitatory_path = os.path.join(get_result_dir(), "cuimc2", "excitatory", "PathwayToGenes.json")
    inhibitory_path = os.path.join(get_result_dir(), "cuimc2", "inhibitory", "PathwayToGenes.json")
    path = excitatory_path if cell_type == "excitatory" else inhibitory_path
    with open(path) as f:
        neuornal_pathways = json.load(f)

    pathways.update(neuornal_pathways)

print(pathways)

In [ ]:
# assert set(pathways.keys()).issubset(set(neuornal_pathways.keys())), "neuornal_pathways and pathways have different keys"

# assert neuornal_pathways.keys() == pathways.keys(), "neuornal_pathways and pathways have different keys"

set(neuornal_pathways.keys()), set(pathways.keys())
for k in pathways.keys():
    if k not in neuornal_pathways:
        print(k)
    else:
        assert neuornal_pathways[k] == pathways[k], "neuornal_pathways and pathways have different values"
# for neuornal_pathway, pathway in zip(neuornal_pathways, pathways):

In [ ]:
for pathway_name, genes in pathways.items():
    if not isinstance(genes, list):
        genes = [genes]

    print(f"Scoring pathway {pathway_name} with {len(genes)} genes")
    genes_symbol = sub_adata.var_names[sub_adata.var_names.isin(genes)].tolist()

    print(f"Found {len(genes_symbol)}/{len(genes)} genes in adata.var_names")
    sc.tl.score_genes(sub_adata, genes_symbol, score_name=f"Pathway_{pathway_name}")
    print(f"Scored pathway {pathway_name} with {len(genes_symbol)} genes")

In [ ]:
import os

# Extract all columns from sub_adata.obs that start with "Pathway"
pathway_cols = [col for col in sub_adata.obs.columns if col.startswith("Pathway")]
pathway_df = sub_adata.obs[pathway_cols]

# Save as pickle, equivalent to RDS in R
pathway_pickle_path = os.path.join(plots_dir, "Pathway.pickle")
pathway_df.to_pickle(pathway_pickle_path)
pathway_df.to_csv(os.path.join(plots_dir, "Pathway.csv"))
pathway_df.head()

In [ ]:
signature_df = pd.read_pickle(os.path.join(plots_dir, "Signature.pickle"))
if not signature_df.columns.isin(sub_adata.obs.columns).all():
    sub_adata.obs = sub_adata.obs.join(signature_df)

pathway_df = pd.read_pickle(os.path.join(plots_dir, "Pathway.pickle"))
if not pathway_df.columns.isin(sub_adata.obs.columns).all():
    sub_adata.obs = sub_adata.obs.join(pathway_df)


In [ ]:
cols = [col for col in sub_adata.obs.columns if col.startswith("Pathway") or col.startswith("Signature") or col in ("Subclass", "Donor ID")]

print(cols)

df_to_save = sub_adata.obs[cols]
df_to_save.to_csv(os.path.join(plots_dir, "pathways_and_signatures.csv.gz"), compression="gzip")

In [ ]:
donor_metadata_cols = ["Donor ID", "Sex", "Age at Death", "PMI", "Overall AD neuropathological Change", "Cognitive Status", "Last MMSE Score", "Severely Affected Donor","Continuous Pseudo-progression Score"]

print(donor_metadata_cols)

df_donor_metadata = sub_adata.obs[donor_metadata_cols].drop_duplicates()
print(df_donor_metadata.columns)
df_donor_metadata.to_csv(os.path.join(plots_dir, "donor_metadata.csv"))